# backprop-pop-outgrad-loop — faded example 3: Complete the pop step that retrieves and removes each node's outgrad

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`. Running the beacon reports progress on the `Backprop: backprop pop-outgrad loop` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The first action when visiting a node is to **pop** its accumulated grad out of the `grads` dict. Popping (rather than peeking with `grads[nid]`) removes the entry so the node cannot be processed twice and a later stray read raises `KeyError`. Because the walk is in reverse-topo order, the popped value is the node's *complete* incoming gradient.

## Faded exercise 3

### Faded — finish the pop step

The driver below is complete except for the line that retrieves the current node's accumulated gradient AND removes it from the `grads` dict. Complete it using the dict method that returns-and-removes. Graph is `y = (a*b)+c`; correct behaviour gives `a.grad==b`, `b.grad==a`, `c.grad==ones`, and leaves no processed-node keys lingering in `grads`.

**Fill in:** Retrieve node `nid`'s gradient from the `grads` dict and remove the entry in one operation (dict pop).

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x
def _add_back0(go, out, x, y):  return go
def _add_back1(go, out, x, y):  return go

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = None  # TODO: pop nid's grad out of the grads dict (return and remove)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
a = MiniTensor(t.randn(3)); b = MiniTensor(t.randn(3)); c = MiniTensor(t.randn(3))
prod_arr = a.array * b.array
prod = MiniTensor(prod_arr, Recipe('mul', (a.array, b.array), {}, {0: a, 1: b}))
y_arr = prod_arr + c.array
y = MiniTensor(y_arr, Recipe('add', (prod_arr, c.array), {}, {0: prod, 1: c}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1,
              ('add', 0): _add_back0, ('add', 1): _add_back1}
backprop(y, t.ones_like(y.array), [y, prod, a, b, c], back_funcs)


def _test():
    assert a.grad is not None and b.grad is not None and c.grad is not None
    assert t.allclose(a.grad, b.array), 'a.grad should equal b'
    assert t.allclose(b.grad, a.array), 'b.grad should equal a'
    assert t.allclose(c.grad, t.ones(3)), 'c.grad should be ones'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x
def _add_back0(go, out, x, y):  return go
def _add_back1(go, out, x, y):  return go

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
a = MiniTensor(t.randn(3)); b = MiniTensor(t.randn(3)); c = MiniTensor(t.randn(3))
prod_arr = a.array * b.array
prod = MiniTensor(prod_arr, Recipe('mul', (a.array, b.array), {}, {0: a, 1: b}))
y_arr = prod_arr + c.array
y = MiniTensor(y_arr, Recipe('add', (prod_arr, c.array), {}, {0: prod, 1: c}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1,
              ('add', 0): _add_back0, ('add', 1): _add_back1}
backprop(y, t.ones_like(y.array), [y, prod, a, b, c], back_funcs)
```
</details>